# Population structure of goats with PCA (solved version)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emmadiepeveen/Evolutionary_Thinking_2026/blob/main/week44/Friday/exercise/pca_goats_solved.ipynb)

Today we look at the genetic structure of **4,653 goats from 144 breeds sampled in 36 countries**. The data come from the ADAPTmap project, described in
[Colli et al. (2018) *Genet Sel Evol* 50:58](https://doi.org/10.1186/s12711-018-0422-x). (You don't have to read the paper, but we will compare our result to their Figure 2.)

Every goat was genotyped at ~53,000 **SNPs** (single nucleotide polymorphisms: positions in the DNA where goats differ by a single letter; explained in Section 1.1). That is far too many numbers to look at directly, so we will use **Principal Component Analysis (PCA)** to summarise them in one plot. In that plot, goats with similar genomes end up close together.

**By the end of this notebook you should be able to:**
1. Explain what a genotype matrix is and how it is stored (PLINK `.bed/.bim/.fam` files).
2. Explain *why* we do quality control (QC) before analysing genetic data, and carry out the standard filters.
3. Explain, without the maths, what a PCA does, and read a PCA plot (what a point, an axis and "% variance explained" mean).
4. Connect the patterns in the plot to evolutionary processes: genetic drift, isolation, migration/gene flow, admixture and the history of goat domestication.

### How to use this notebook
**This is the solved version: all the code is already filled in.** You can simply run all cells (*Run All*, or `Shift + Enter` cell by cell) and focus on the concepts, the plots and the questions. If you'd like to practise writing the code yourself, use `pca_goats.ipynb` instead.

* Run the cells **from top to bottom**. Each cell can use the variables made in earlier cells.
* **💻 Code** sections explain what the next cell computes. Open **💡 Syntax explained** if you want to understand how the code works.
* **❓ Question** blocks are for thinking and discussing with your neighbour. Write your answer down (or discuss it) **before** clicking *Reveal answer*.

## 0. Setup

### 0.1 (Optional) Install the packages

A notebook runs on a **kernel**, the Python installation that executes your code. If a package isn't installed in that Python, you get `ModuleNotFoundError: No module named ...`.

The cell below checks which packages are missing and installs **only those**, directly into the Python that runs this notebook. That way you don't need to set up conda or environments yourself.

* **Run it** if you are on Google Colab, or if you get a `ModuleNotFoundError` later on.
* **Skip it** if you already created the course environment from `env.yaml`.
* It is safe to run more than once. If nothing is missing, it does nothing.
* If an import still fails after installing, restart the kernel (*Kernel → Restart*, or *Runtime → Restart session* in Colab) and run the notebook again from the top.

In [ ]:
# OPTIONAL: installs any missing packages into the Python that runs this notebook
import importlib.util
import subprocess
import sys

# "name you import" : "name you install with pip"
required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "bed_reader": "bed-reader",
}

missing = [pip_name for module, pip_name in required.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Installing:", ", ".join(missing), "(this can take a minute)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
    print("Done! All packages are installed.")
else:
    print("All packages are already installed - nothing to do.")

### 0.2 The packages we use

You probably know some of these already. The table says what each package is for and which of its features we use today.

| Package | What it is | What we use it for today |
|---|---|---|
| **numpy** (`np`) | Fast arrays (vectors/matrices) of numbers. Almost all scientific Python is built on it. | Our genotype matrix `X`. Functions like `np.isnan`, `np.nanmean`, `np.where`. |
| **pandas** (`pd`) | Tables, called *DataFrames*. Think of an Excel sheet or an R `data.frame`. | The information about each goat (breed, country) and each SNP (chromosome, position). |
| **bed-reader** | A small package that reads PLINK binary genotype files (`.bed`) into a numpy array. | Loading the genotypes. |
| **scikit-learn** (`sklearn`) | The standard Python library for machine learning and statistics. | `PCA`, and later `SimpleImputer`. |
| **matplotlib** (`plt`) | The standard plotting library. | Scatter plots, histograms, bar plots. |

<details>
<summary><b>🐍 Python refresher: syntax used in this notebook (open if you're rusty)</b></summary>

**numpy arrays**
```python
X.shape          # (number of rows, number of columns)
X[0, 5]          # value in row 0, column 5   (Python counts from 0!)
X[:5, :3]        # first 5 rows, first 3 columns   (":" means "everything")
X[keep, :]       # keep only the rows where the boolean array `keep` is True
X[:, keep]       # keep only the columns where `keep` is True
np.isnan(X)      # array of True/False: True where the value is missing (NaN = "Not a Number")
```

**Comparisons give True/False arrays ("boolean masks")**
```python
values <= 0.05                     # True where the value is at most 0.05
(a >= 1) & (a <= 29)               # AND - the parentheses are required!
(a < 1) | (a > 29)                 # OR
~mask                              # NOT - flips every True to False and every False to True
```
For arrays and pandas columns you must use `&`, `|` and `~`. Python's normal `and`, `or` and `not` only work on a single True/False value and give an error on arrays.

`True` counts as 1 and `False` as 0, so `mask.sum()` counts the Trues and `mask.mean()` gives the *fraction* of Trues.

**Methods need parentheses**
```python
df["column"].value_counts()        # () = run the method -> you get the result
df["column"].value_counts          # no () = the method itself, it doesn't run!
X.shape                            # attributes (stored values) have no ()
```
If you see something like `<bound method ...>` printed, you forgot the `()`.

**pandas DataFrames**
```python
df["column"]                       # one column (a "Series")
df["column"].value_counts()        # how often each value occurs
df["column"].nunique()             # number of different values
df[df["breed"] == "SAA"]           # only the rows where breed is SAA
df["column"].isin(["A", "B"])      # True where the value is A or B
df["new"] = ...                    # add a new column
```

**f-strings** (for printing nicely)
```python
x = 3.14159
print(f"x is about {x:.2f}")       # -> x is about 3.14   (.2f = 2 decimals)
```
</details>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bed_reader import open_bed
from sklearn.decomposition import PCA


def check(condition, hint_if_wrong):
    # Small helper for the check cells: prints a tick, or stops with a hint about what is wrong
    if not condition:
        raise AssertionError("Not quite yet - " + hint_if_wrong)
    print("✅ Looks good!")


def check_todo(**variables):
    # Used at the start of some cells: stops with a clear message if a TODO they need still contains the placeholder ...
    for name, value in variables.items():
        if value is ... or (isinstance(value, pd.Series) and value.iloc[0] is ...):
            raise RuntimeError(f"`{name}` is not ready yet: it still contains the placeholder `...` (Ellipsis). "
                               f"Finish the TODO that makes `{name}` (replace the `...` with your code), "
                               f"run that cell, and then run this cell again.")


print("Imports worked!")

## 1. The data

### 1.1 Background: what is a SNP?

Before we look at the files, we need to understand what was actually measured in these goats.

**DNA varies a little between individuals.** A goat's genome is a DNA sequence of about 3 billion "letters" (bases: `A`, `C`, `G`, `T`). If you line up the genomes of two goats, almost every letter is the same. But here and there, at some positions, some goats carry one letter and other goats another.

**A SNP is one such position.** SNP stands for **S**ingle **N**ucleotide **P**olymorphism (pronounced "snip"):
* *single nucleotide*: it concerns one single letter of the DNA;
* *polymorphism* ("many forms"): more than one version exists in the population.

Here is a small stretch of DNA from two goats. Every goat has two copies of each chromosome, one from its mother and one from its father:

```
                                  this position differs
                                            ↓
Goat 1, copy from mother:   ... C T G A T C A G T C C A ...
Goat 1, copy from father:   ... C T G A T C A G T C C A ...
Goat 2, copy from mother:   ... C T G A T C G G T C C A ...
Goat 2, copy from father:   ... C T G A T C A G T C C A ...
```

At the marked position, three of the four chromosome copies carry an `A` and one carries a `G`. This position is a SNP. Its two versions, `A` and `G`, are called the two **alleles**. All the other positions shown are identical in everyone, so they tell us nothing about how these goats differ.

**Where do SNPs come from?** Every SNP started as a **mutation**: a copying error when DNA was replicated in an egg or sperm cell of one goat (or a wild ancestor) long ago. That goat passed the new allele on to its offspring, who passed it on to theirs, and so on. Over many generations, the frequency of the new allele changes. Mostly this happens by chance (**genetic drift**), sometimes through **selection**, and alleles also move between populations through **migration**. The same position almost never mutates twice, so nearly all SNPs have exactly **two** alleles.

**Why are SNPs useful for studying evolution?**
* **They are inherited.** You get your alleles from your parents, so related individuals share more alleles.
* **Their frequencies differ between populations.** When populations are isolated, drift changes their allele frequencies independently. At one SNP, an allele might be at 80% in one breed and 30% in another. A single SNP tells you little: both alleles usually occur in most populations. But **thousands** of SNPs together form a very precise picture of ancestry, a bit like a genetic fingerprint of a population.
* **There are millions of them**, and they are cheap to measure.

**How were they measured?** Sequencing the whole genome of 4,653 goats would be expensive. Instead, the ADAPTmap goats were genotyped with a **SNP chip** (the Illumina *Goat SNP50 BeadChip*). That is a small device that tests a fixed list of ~50,000 known SNPs spread across the genome, and reports which alleles each goat carries at each of them. One caveat: the SNPs on the chip were discovered in a few (mostly European, commercial) breeds, so they don't represent the variation in every breed equally well. This is called **ascertainment bias**. We'll come back to it in the quality control.

**Genotypes.** Since a goat has two copies of each autosome, it has **two alleles at every SNP**. Together these make up its **genotype**. At a SNP with alleles `A` and `G`, a goat is either `AA` or `GG` (two identical copies: **homozygous**) or `AG` (two different copies: **heterozygous**). It doesn't matter which copy came from which parent. For the computer, we store the genotype as **the number of copies of allele 1**:

| Genotype (allele 1 = `A`, allele 2 = `G`) | Homozygous / heterozygous | Copies of allele 1 | Stored as |
|---|---|---|---|
| A A | homozygous | 2 | `2` |
| A G | heterozygous | 1 | `1` |
| G G | homozygous | 0 | `0` |
| genotyping failed | ? | ? | `NaN` (missing) |

So in the end, all our data is just a big table of 0s, 1s and 2s: one row per goat, one column per SNP.

### ❓ Question: Check your understanding of SNPs

At a SNP, allele 1 is `G` and allele 2 is `T`.
1. Goat X is heterozygous. Goat Y carries two copies of `T`. Goat Z carries two copies of `G`. Which numbers are stored for X, Y and Z?
2. What is the frequency of allele `G` among these three goats? (Hint: how many chromosome copies are there in total?)
3. Why can a single SNP hardly tell you which population a goat comes from, while 50,000 SNPs can?

<details>
<summary><b>✅ Reveal answer</b></summary>

1. X (`GT`) → `1`. Y (`TT`) → `0`. Z (`GG`) → `2`.
2. Three diploid goats carry 3 × 2 = 6 copies of the chromosome. Of these, 1 + 0 + 2 = 3 carry `G`, so the frequency of `G` is 3/6 = **0.5**. Notice that this is the **mean genotype divided by 2**: (1 + 0 + 2) / 3 / 2 = 0.5. We'll use exactly this trick in the quality control.
3. At most SNPs, both alleles occur in most populations. Populations differ in **how common** each allele is, not in having completely different alleles. So one genotype is weak evidence, a bit like guessing someone's nationality from one answer to one question. But every SNP adds a little evidence, and the SNPs on different chromosomes are inherited (largely) independently. Combined over thousands of SNPs, these small differences give a strong and consistent signal. That's exactly what the PCA will summarise.

</details>

### 1.2 Getting the data

The cell below looks for a `data/` folder containing `snps.bed`, either next to this notebook or one folder up (where it lives in the course repository). If it can't find it, for example on Colab, it downloads the data from GitHub (~45 MB) and unzips it. You don't need to change anything here.

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

DATA_URLS = [
    "https://github.com/emmadiepeveen/Evolutionary_Thinking_2026/raw/main/week44/Friday/data.zip",
    "https://github.com/Bjarke-M/Evolutionary_Thinking_2024/raw/fb17582a10b81c6b8ef9641222eb012a69c227fb/week44/Friday/data.zip",
]


def find_or_download_data():
    for folder in [Path("data"), Path("../data")]:
        if (folder / "snps.bed").exists():
            print(f"Found the data in: {folder.resolve()}")
            return folder

    print("Data not found locally - downloading (~45 MB, may take a minute)...")
    for url in DATA_URLS:
        try:
            urllib.request.urlretrieve(url, "goats.zip")
            break
        except Exception as error:
            print(f"  Could not download from {url}: {error}")
    else:
        raise RuntimeError("Download failed. Download data.zip from the course GitHub page "
                           "and unzip it in the same folder as this notebook.")
    with zipfile.ZipFile("goats.zip") as z:
        z.extractall(".")
    print("Downloaded and unzipped.")
    return Path("data")


data_dir = find_or_download_data()

### 1.3 PLINK files: `.fam`, `.bim` and `.bed`

The data are stored in **PLINK binary format**, the most common format for SNP data. It is always a trio of files with the same name:

| File | Contents | One line per... | Columns |
|---|---|---|---|
| `snps.fam` | Information about the **individuals** (text) | goat | family ID (*here: the breed code*), individual ID, father, mother, sex (1 = male, 2 = female, 0 = unknown), phenotype (-9 = missing) |
| `snps.bim` | Information about the **SNPs** (text) | SNP | chromosome, SNP name, genetic position (cM), base-pair position, allele 1, allele 2 |
| `snps.bed` | The **genotypes** themselves (binary) | - | every goat × every SNP, packed 4 genotypes per byte, so you can't open it in a text editor |

> ⚠️ Confusingly, there is also a completely different file type called "BED" (from the UCSC genome browser) that stores genomic intervals. Same extension, nothing to do with PLINK.

The `.fam` and `.bim` files are plain text, so we can read them with pandas:
* `sep=r"\s+"` means "columns are separated by any whitespace (spaces or tabs)".
* `header=None` because the files have no header line, so we give the column names ourselves with `names=`.

In [ ]:
fam = pd.read_csv(data_dir / "snps.fam", sep=r"\s+", header=None,
                  names=["breed", "iid", "father", "mother", "sex", "phenotype"])
bim = pd.read_csv(data_dir / "snps.bim", sep=r"\s+", header=None,
                  names=["chrom", "snp", "cM", "pos", "allele_1", "allele_2"])

print(f"{len(fam)} goats and {len(bim)} SNPs")
fam.head()

In [ ]:
bim.head()

### ❓ Question: Reading the .bim file

Each row above is one SNP. Look at the row with SNP name `snp10412-scaffold1372-579082`: its `allele_1` is `0` instead of a DNA letter. What do you think that means? Is this SNP useful for studying how goats differ? (Also: why do the first rows have chromosome `0` and position `0`?)

<details>
<summary><b>✅ Reveal answer</b></summary>

PLINK writes `0` for an allele that was **never observed**. Every goat in the dataset that was successfully genotyped carries `G` at this SNP, so in our goats it isn't polymorphic at all (it is **monomorphic**). The chip most likely includes it because it was variable in the animals used to design the chip (ascertainment bias again!). A SNP where everyone has the same genotype carries **no information** about differences between goats. SNPs like this are removed by the minor allele frequency filter in Section 3. (This particular one is already removed earlier, because it is on chromosome `0`.)

Chromosome `0` and position `0` mean the SNP's location in the goat reference genome is unknown (it is "unplaced"; the name refers to a piece of an older genome assembly, a *scaffold*). We'll remove these in Section 1.4.

</details>

### 💻 Code: Get to know the goats

How many different breeds are there, and which breeds have the most goats?

Store the number of breeds in `n_breeds` and show the 10 largest breeds.

*(You'll notice some breed codes contain an `x`, e.g. `SAAxCRE`. These are **crossbreeds**: here, Saanen × Creole. Keep them in mind; we'll come back to them!)*

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* `df["column"].nunique()` counts the number of *different* values in a column.
* `df["column"].value_counts()` counts how often each value occurs, sorted from most to least common.
* `.head(10)` keeps the first 10 rows.

</details>

In [ ]:
n_breeds = fam["breed"].nunique()   # number of different breeds
print("Number of breeds:", n_breeds)

# show the 10 breeds with the most goats
fam["breed"].value_counts().head(10)

In [ ]:
check(n_breeds == 144, "n_breeds should be a single number: the count of *different* breed codes.")

### 💻 Code: Where was each goat sampled?

The individual IDs look like `IT_ALP0001`. The two letters before the `_` are the **country code** of where the goat was sampled (IT = Italy, ET = Ethiopia, PK = Pakistan, ...).

1. Make a new column `fam["country"]` with the part of the ID **before** the underscore.
2. Use the dictionary `country_to_continent` (given below) to make a column `fam["continent"]`.

We will use the continent later to colour the PCA plot.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* pandas has string tools under `.str`. They apply a string operation to **every** row of a column at once:
  ```python
  fam["iid"].str.split("_")          # "IT_ALP0001" -> ["IT", "ALP0001"]
  fam["iid"].str.split("_").str[0]   # take element 0 of each list -> "IT"
  ```
  (`fam["iid"].str[:2]`, the first two characters, would also work here.)
* `.map(dictionary)` replaces each value with the value it points to in the dictionary:
  ```python
  pd.Series(["IT", "PK"]).map({"IT": "Europe", "PK": "Asia"})   # -> "Europe", "Asia"
  ```

</details>

In [ ]:
country_to_continent = {
    "BF": "Africa", "BI": "Africa", "CM": "Africa", "EG": "Africa", "ET": "Africa", "KE": "Africa",
    "MA": "Africa", "MG": "Africa", "ML": "Africa", "MW": "Africa", "MZ": "Africa", "NG": "Africa",
    "TN": "Africa", "TZ": "Africa", "UG": "Africa", "ZA": "Africa", "ZW": "Africa",
    "CH": "Europe", "DK": "Europe", "ES": "Europe", "FI": "Europe", "FR": "Europe", "GB": "Europe",
    "IE": "Europe", "IS": "Europe", "IT": "Europe", "NL": "Europe", "RO": "Europe",
    "IR": "Asia", "PK": "Asia", "TR": "Asia",
    "AR": "Americas", "BR": "Americas", "US": "Americas",
    "AU": "Oceania", "NZ": "Oceania",
}

fam["country"] = fam["iid"].str.split("_").str[0]
fam["continent"] = fam["country"].map(country_to_continent)

fam["continent"].value_counts()

In [ ]:
check("country" in fam and fam["country"].nunique() == 36, "fam['country'] should contain 36 different two-letter codes.")
check("continent" in fam and fam["continent"].notna().all(), "every goat should get a continent (no NaN values).")

### 1.4 Chromosomes

Each SNP sits on a chromosome. Goats have **29 pairs of autosomes** plus the sex chromosomes (X and Y). In this dataset the `.bim` file uses these chromosome codes:
* `1`–`29`: the autosomes
* `30`: the X chromosome
* `0`: SNPs that could not be placed on any chromosome of the reference genome (position unknown)

### 💻 Code: How many SNPs are on each chromosome?

Count the number of SNPs per chromosome code, sorted by chromosome number (0, 1, 2, ..., 30), not by count.

Look at the result: is there a pattern in how many SNPs each autosome has? (Chromosome 1 is the largest goat chromosome and 29 the smallest.)

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* The chromosome of each SNP is in the column `bim["chrom"]`.
* `.value_counts()` counts how often each value occurs. Don't forget the `()`: without them you get the method itself (`<bound method ...>`) instead of the counts.
* `value_counts()` sorts by *count*. Add `.sort_index()` to sort by the values themselves (here the chromosome numbers). You can chain them: `.value_counts().sort_index()`.

</details>

In [ ]:
snps_per_chrom = bim["chrom"].value_counts().sort_index()
snps_per_chrom

In [ ]:
check(isinstance(snps_per_chrom, pd.Series),
      "snps_per_chrom should be a table of counts. If you see '<bound method ...>', you forgot the () after value_counts.")
check(list(snps_per_chrom.index) == sorted(snps_per_chrom.index),
      "the counts should be sorted by chromosome number (0, 1, 2, ..., 30). Did you add .sort_index()?")
check(snps_per_chrom[1] == 3258, "chromosome 1 should have 3,258 SNPs. Did you count the 'chrom' column of bim?")

### 💻 Code: Select the autosomal SNPs

We only want to keep the SNPs on the **autosomes** (chromosomes 1 to 29). Make a **boolean mask** `is_autosome`: an array with one `True`/`False` per SNP, `True` if the SNP is on chromosome 1–29.

We will use this mask in the next section to load only the autosomal genotypes.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* The chromosome of each SNP is in the column `bim["chrom"]`.
* To test whether values lie in a range, use the method `.between(lowest, highest)`. Both ends are included:
  ```python
  pd.Series([0, 5, 30]).between(1, 29)                    # -> False, True, False
  ```
* You can also write the same test with two comparisons. This gives **exactly the same** True/False result, so use whichever you find clearer:
  ```python
  s = pd.Series([0, 5, 30])
  (s >= 1) & (s <= 29)                                    # -> False, True, False
  ```
  Use `&` (not `and`), and put each comparison in parentheses.
* Add `.to_numpy()` at the end to turn the result into a numpy array, which we need to select columns of the numpy matrix `X` in the next section.

</details>

In [ ]:
is_autosome = bim["chrom"].between(1, 29).to_numpy()

# This is equally correct (it gives exactly the same True/False values):
# is_autosome = ((bim["chrom"] >= 1) & (bim["chrom"] <= 29)).to_numpy()

check_todo(is_autosome=is_autosome)
print(f"{is_autosome.sum()} of {len(is_autosome)} SNPs are autosomal")

In [ ]:
check(isinstance(is_autosome, np.ndarray) and is_autosome.dtype == bool and is_autosome.sum() == 49953,
      "is_autosome should be a numpy array of True/False with 49,953 Trues. Did you forget .to_numpy(), or include chromosome 0 or 30?")

### ❓ Question: Why only autosomes?

Why do we throw away the SNPs on the X chromosome (`30`) and the unplaced SNPs (`0`)? In general, why would you also remove mitochondrial and Y-chromosome SNPs before a PCA of population structure? *(Think back to today's lecture on the X chromosome!)*

<details>
<summary><b>✅ Reveal answer</b></summary>

We want to capture **genome-wide ancestry**, and the autosomes all follow the same simple rules: diploid in both sexes, inherited from both parents, recombining. The other parts of the genome break those rules:

* **X chromosome:** males have only **one** X (they are *hemizygous*), females have two. So the "same" genotype means something different in the two sexes, and males can look falsely homozygous. Sex is unknown for most goats here (`sex = 0` in the `.fam` file), so we can't even correct for it. The X also has its own evolutionary history. Its effective population size is about ¾ of the autosomes' (fewer copies in the population). It experiences mutation and selection differently: it spends only a third of its time in males, so it sees less male-biased mutation, and it is exposed to things like **meiotic drive**. Including it would mix sex differences and X-specific history into the ancestry signal.
* **Unplaced SNPs (chromosome 0):** we don't know where in the genome they are. They could sit on the X/Y, in mitochondrial DNA or in duplicated/repetitive regions, where genotyping is often unreliable.
* **Mitochondrial DNA** (none in this dataset, but it is common in other datasets): inherited from the mother only, haploid and with no recombination. It traces only the **maternal lineage**, so it tells a different (single-gene-tree) story.
* **Y chromosome:** only in males and inherited only from father to son, so it traces only the **paternal lineage**.

These regions are interesting in their own right (see the optional exercises), but they should be analysed separately.

</details>

## 2. Loading the genotypes

Now we read the genotypes from the `.bed` file with **bed-reader**:

* `open_bed(...)` opens the file and reads the SNP/sample information, but **not yet** the (large) genotype data.
* `.read(...)` actually loads the genotypes into a numpy array:
  * `index=np.s_[:, is_autosome]` says *which part* to read. (Careful: if `is_autosome` were still the placeholder `...`, numpy would silently read **everything**, because `...` means "all" in numpy indexing. That's why the cell starts with `check_todo`.)
  * `np.s_[...]` is just a way to write "square-bracket indexing" as an argument: `[:, is_autosome]` means "all goats (rows), and only the columns (SNPs) where `is_autosome` is True".
  * `dtype="float32"` stores the numbers as 32-bit decimals. That uses half the memory of the default (64-bit) and still lets us store missing values as `NaN`.

Afterwards we also keep only the autosomal rows of `bim`, so `bim` keeps describing the columns of `X`.

This takes a few seconds and uses ~1 GB of memory.

In [ ]:
check_todo(is_autosome=is_autosome)     # stops with a message if the TODO "Select the autosomal SNPs" isn't finished

bed = open_bed(data_dir / "snps.bed")
X = bed.read(index=np.s_[:, is_autosome], dtype="float32")
bim = bim[is_autosome].reset_index(drop=True)

print("Shape of X (goats, SNPs):", X.shape)
X[:5, :8]   # the first 5 goats and the first 8 SNPs

### The genotype matrix

`X` is the central object of today's exercise. Each **row is a goat** and each **column is a SNP**:

```
                    SNP 1   SNP 2   SNP 3   ...   SNP 49953      <- described by the rows of `bim`
goat ET_ABR0001       2       1       0     ...       1
goat ET_ABR0002       2      NaN      0     ...       2
goat ET_ABR0003       1       1       0     ...       1
...                                                               <- described by the rows of `fam`
```

> ⚠️ **Keep everything in sync.** Row *i* of `X` is the goat in row *i* of `fam`, and column *j* of `X` is the SNP in row *j* of `bim`. Whenever we remove goats or SNPs from `X`, we must remove the same ones from `fam` or `bim`. Otherwise the labels no longer match the data.

### 💻 Code: How much data is missing?

What **fraction** of all genotypes in `X` is missing (`NaN`)? Store it in `missing_fraction` and print it as a percentage.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* `np.isnan(X)` gives a True/False matrix of the same shape as `X`: True where the genotype is missing.
* `True` counts as 1 and `False` as 0, so the **mean** of a True/False array is the fraction of Trues. Use `.mean()`.

</details>

In [ ]:
missing_fraction = np.isnan(X).mean()

check_todo(missing_fraction=missing_fraction)
print(f"{missing_fraction * 100:.2f}% of all genotypes are missing")

In [ ]:
check_todo(missing_fraction=missing_fraction)
check(0.023 < missing_fraction < 0.026, "you should get a single number around 0.024 (= 2.4%).")

## 3. Quality control (QC)

SNP genotyping is not perfect. Some DNA samples are degraded or contaminated, and some SNPs simply don't work well in some breeds. Bad data can create **fake structure** in a PCA: for example, a cluster of goats that were genotyped badly rather than goats that are genetically related. So before any analysis we apply some standard filters:

1. Remove **goats** with more than 5% missing genotypes.
2. Remove **SNPs** with more than 5% missing genotypes.
3. Remove **SNPs** with a minor allele frequency (MAF) below 5%.

### Rows, columns and `axis`
Many numpy functions take an `axis` argument that says **in which direction to summarise**:

```
np.isnan(X).mean(axis=1)   ->  one value per ROW     = per goat   (average across the SNPs)
np.isnan(X).mean(axis=0)   ->  one value per COLUMN  = per SNP    (average across the goats)
```
A way to remember it: `axis` is the dimension that gets *squashed*. `axis=0` squashes the rows together, leaving one number per column.

To **filter**, we build a True/False mask (one value per goat or per SNP) and use it to index:
```python
X[keep_goats, :]     # keep rows    (goats) where keep_goats is True
X[:, keep_snps]      # keep columns (SNPs)  where keep_snps is True
fam[keep_goats]      # the same goats in the fam table
```

### 💻 Code: Filter goats with too much missing data

1. Compute the fraction of missing genotypes **for each goat**: `missing_per_goat`.
2. Make a mask `keep_goats` that is True for goats with **at most 5%** missing genotypes.

The rest of the cell is given. It applies your mask to **both** `X` and `fam`, so they stay in sync (row *i* of `X` must still be the goat in row *i* of `fam`).

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* Combine `np.isnan(X)` with `.mean(axis=...)`, like in the explanation above. Goats are the **rows**, so you want one value per row: `axis=1`.
* "At most 5%" means `<= 0.05`. Comparing an array with a number gives one True/False per element.
* In the given part, `~` means **NOT**: it flips every True to False and every False to True.
  ```python
  keep_goats  = [True,  True,  False, True ]    # True = goat passes the filter
  ~keep_goats = [False, False, True,  False]    # True = goat is REMOVED
  ```
  So `(~keep_goats).sum()` counts the removed goats. (Python's normal `not` doesn't work on arrays; use `~`.)
* `fam[keep_goats]` keeps the rows of the table where the mask is True. `.reset_index(drop=True)` renumbers the rows 0, 1, 2, ... again.

</details>

In [ ]:
missing_per_goat = np.isnan(X).mean(axis=1)      # one value per goat
keep_goats = missing_per_goat <= 0.05            # True/False per goat

# (given) apply the filter to X and fam
check_todo(keep_goats=keep_goats)
X = X[keep_goats, :]
fam = fam[keep_goats].reset_index(drop=True)

print(f"Removed {(~keep_goats).sum()} goats, {X.shape[0]} goats left")

In [ ]:
check(X.shape[0] == 4451, "X should have 4,451 goats left. Did you use axis=1 and <= 0.05?")
check(len(fam) == X.shape[0], "fam must have the same number of rows as X. Did you filter fam too?")

In [ ]:
# How was the missingness distributed? (Uses missing_per_goat from before filtering.)
plt.figure(figsize=(7, 3))
plt.hist(missing_per_goat, bins=100)
plt.axvline(0.05, color="red", linestyle="--", label="5% threshold")
plt.yscale("log")
plt.xlabel("Fraction of missing genotypes per goat")
plt.ylabel("Number of goats (log scale)")
plt.legend()
plt.show()

### 💻 Code: Filter SNPs with too much missing data

Now do the same for the **SNPs**: make a mask `keep_snps` that keeps every SNP that is missing in **at most 5%** of the (remaining) goats. The given part applies it to **both** `X` and `bim`.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* Same as for the goats, but SNPs are the **columns**: `np.isnan(X).mean(axis=0)` gives one value per column.
* Then `<= 0.05` to get the True/False mask.

</details>

In [ ]:
missing_per_snp = np.isnan(X).mean(axis=0)       # one value per SNP
keep_snps = missing_per_snp <= 0.05              # True/False per SNP

# (given) apply the filter to X and bim
check_todo(keep_snps=keep_snps)
X = X[:, keep_snps]
bim = bim[keep_snps].reset_index(drop=True)

print(f"Removed {(~keep_snps).sum()} SNPs, {X.shape[1]} SNPs left")

In [ ]:
check(X.shape[1] == 48062, "X should have 48,062 SNPs left. Did you use axis=0 and filter the columns?")
check(len(bim) == X.shape[1], "bim must have one row per column of X. Did you filter bim too?")

### ❓ Question: Why filter on missingness?

Why do we remove goats with >5% missing genotypes and SNPs with >5% missing genotypes? What could go wrong in the PCA if we kept them?

<details>
<summary><b>✅ Reveal answer</b></summary>

* **Missingness is a sign of bad quality.** A goat with many failed genotypes probably had poor DNA (degraded, contaminated, low concentration). Its *non*-missing genotypes are then also less reliable. A SNP that fails in many goats is probably badly designed, for example in a repetitive region or close to another variant that disturbs the probe.
* **Missingness is often not random.** SNP chips are designed using a few (mostly European, commercial) breeds. In distant breeds, some probes work worse, so missingness itself can correlate with breed. If we keep such data, the PCA might separate goats by *how well they were genotyped* rather than by ancestry: a fake (technical) population structure.
* **Fewer values to impute.** Every missing value must be filled in before the PCA (Section 4), and every filled-in value is a guess.

The 5% threshold is a common compromise: strict enough to remove clearly bad data, loose enough to keep most goats and SNPs. (Here we lost ~200 goats and ~1,900 SNPs.) Note that the order matters a little: we filtered goats first, so the SNP missingness was computed using only the good goats.

</details>

### Minor allele frequency (MAF)

The **allele frequency** *p* of allele 1 at a SNP is the fraction of all chromosome copies in the sample that carry allele 1. Each goat contributes **two** copies, and its genotype (0, 1 or 2) counts how many of them are allele 1, so:

$$p = \frac{\text{total number of allele-1 copies}}{\text{total number of copies}} = \frac{\sum \text{genotypes}}{2 \times \text{number of goats}} = \frac{\text{mean genotype}}{2}$$

The **minor allele** is whichever of the two alleles is rarer, so the **minor allele frequency** is

$$\text{MAF} = \min(p,\ 1 - p)$$

which is always between 0 and 0.5.

### 💻 Code: Filter SNPs with a low minor allele frequency

1. Compute the frequency `p` of allele 1 for every SNP. Be careful with the missing values!
2. Compute the minor allele frequency `maf` for every SNP.
3. Make a mask `keep_snps` that keeps only SNPs with MAF ≥ 0.05.

The given part applies it to **both** `X` and `bim`.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* `np.mean` returns `nan` as soon as **one** value is missing. `np.nanmean(X, axis=0)` computes the mean of every column (SNP) while **ignoring** the NaNs. Divide it by 2 to get *p* (see the formula above).
* `np.minimum(a, b)` compares two arrays **element by element** and returns the smaller value at each position. Here: `np.minimum(p, 1 - p)`. Don't confuse it with `np.min(a)`, which returns the single smallest value of the whole array!
* "MAF ≥ 0.05" in Python is `maf >= 0.05`.

</details>

In [ ]:
p = np.nanmean(X, axis=0) / 2        # frequency of allele 1, one value per SNP
maf = np.minimum(p, 1 - p)           # minor allele frequency, one value per SNP
keep_snps = maf >= 0.05              # True/False per SNP

# (given) apply the filter to X and bim
check_todo(keep_snps=keep_snps)
X = X[:, keep_snps]
bim = bim[keep_snps].reset_index(drop=True)

print(f"Removed {(~keep_snps).sum()} SNPs, {X.shape[1]} SNPs left")

In [ ]:
check(X.shape[1] == 47995, "X should have 47,995 SNPs left. Did you use np.nanmean, divide by 2 and use np.minimum?")
check(len(bim) == X.shape[1], "bim must have one row per column of X. Did you filter bim too?")

### ❓ Question: Why filter on MAF?

What is the MAF, and why do we divide by 2 when we calculate it? Why remove SNPs with a MAF below 5%?

<details>
<summary><b>✅ Reveal answer</b></summary>

* **MAF** = the frequency of the rarer allele at a SNP (between 0 and 0.5).
* **Divide by 2** because goats are **diploid**: each goat carries two copies of every autosomal SNP, and the genotype (0/1/2) counts the copies of allele 1. The mean genotype is therefore between 0 and 2, and dividing by 2 turns it into a frequency between 0 and 1.
* **Why remove rare SNPs?**
  * Rare alleles are found in only a few goats, so a **genotyping error** has a relatively large effect. Some apparent rare alleles *are* errors.
  * A SNP where almost every goat has the same genotype has very little **variance**, so it carries almost no information about how goats relate to each other. PCA is driven by SNPs whose allele frequencies **differ between populations**, and those are mostly common SNPs.
  * Rare variants are often **private** to one family or breed, so they add noise rather than shared-ancestry signal.
* (The old version of this notebook wrote `(p >= 0.05) & (p <= 0.95)`. That is exactly the same filter as `min(p, 1-p) >= 0.05`.)

</details>

In [ ]:
print(f"After QC: {X.shape[0]} goats x {X.shape[1]} SNPs  (we started with {bed.iid_count} goats x {bed.sid_count} SNPs)")

## 4. Imputation: filling in the missing genotypes

Even after QC, a small fraction of genotypes is still missing. PCA is linear algebra (matrix multiplication), and that simply cannot handle `NaN`s. We have two options: throw away every goat or SNP with **any** missing value (which would remove almost everything), or **impute** them, i.e. fill in a reasonable guess.

The simplest approach is **mean imputation**: replace a missing genotype at SNP *j* by the **mean genotype at that SNP** (= 2*p*<sub>*j*</sub>). That is the genotype we would *expect* for a random goat from the whole dataset, if we know nothing else about it.

### 💻 Code: Mean imputation

1. Compute the mean genotype of every SNP (column), ignoring missing values: `snp_means`.
2. Make a new matrix `X_imputed` that is equal to `X`, except that each `NaN` is replaced by the mean of its SNP (column).

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* `np.where(condition, value_if_true, value_if_false)` works element by element:
  ```python
  np.where([True, False, True], 99, [1, 2, 3])   # -> [99, 2, 99]
  ```
* **Broadcasting:** if `snp_means` has shape `(n_snps,)` and `X` has shape `(n_goats, n_snps)`, numpy automatically "stretches" `snp_means` across every row. So for each missing value, `np.where` picks the mean of **its own column**.
* *(scikit-learn can do the same with `SimpleImputer(strategy="mean").fit_transform(X)`. It also works column by column, which is one reason why goats must be the rows!)*

</details>

In [ ]:
snp_means = np.nanmean(X, axis=0)                     # one value per SNP
X_imputed = np.where(np.isnan(X), snp_means, X)

check_todo(X_imputed=X_imputed)
print("Missing values left:", np.isnan(X_imputed).sum())

In [ ]:
check_todo(X_imputed=X_imputed)
check(X_imputed.shape == X.shape, "X_imputed should have the same shape as X.")
check(not np.isnan(X_imputed).any(), "X_imputed should not contain any NaN values.")
check(np.allclose(np.nanmean(X, axis=0), X_imputed.mean(axis=0), atol=1e-4),
      "imputing with the column mean should leave each SNP's mean unchanged. Did you use axis=0?")

### ❓ Question: Imputation

Why do we have to impute before running the PCA? What do we implicitly assume when we fill in the **mean** genotype, and where in the PCA plot will a goat with a lot of imputed values tend to end up?

<details>
<summary><b>✅ Reveal answer</b></summary>

* PCA works on a **complete** matrix. Missing values break the calculations. Dropping every SNP or goat with any missing value would throw away almost all the data, so we impute instead.
* Mean imputation fills in the genotype of an "average goat of the whole dataset". It ignores which **breed** the goat belongs to, and the information in nearby SNPs, which are correlated through linkage disequilibrium. A Pakistani goat with a missing genotype gets the global average, which is partly based on European and African goats.
* Imputed values pull a goat **towards the centre** of the PCA plot (towards the global average). Here only ~0.3% of genotypes are imputed, so the effect is tiny. That's another reason to filter out goats with a lot of missing data first.
* Better methods exist: impute using the goat's own population, or use correlations with neighbouring SNPs (LD-based imputation with tools like Beagle). These matter much more when there is more missing data.

</details>

## 5. Principal Component Analysis

### 5.1 The idea, without the maths

Each goat is now described by ~48,000 numbers, so each goat is a point in a 48,000-dimensional space. We can't plot that. PCA finds **new axes** (directions) through this cloud of points:

* **PC1** is the direction along which the goats differ **the most** (the largest variance).
* **PC2** is the direction with the most remaining variance, perpendicular to PC1. Then PC3, and so on.
* Each PC is a weighted combination of all SNPs. The **PC score** of a goat is its position along that axis.
* The **% variance explained** by a PC is how much of the total variation among goats lies along that axis.

If the first few PCs capture the main differences, plotting PC1 against PC2 gives a good 2D "map" of the data.

The toy example below shows this with only two measured variables, so we can see the original data and the new axes together. The arrows point along PC1 and PC2. Note that PC1 runs along the long direction of the cloud.

In [ ]:
# Toy example: 300 "goats" measured for two correlated traits
rng = np.random.default_rng(1)
length = rng.normal(100, 10, 300)
weight = 0.6 * length + rng.normal(0, 3, 300)
toy = np.column_stack([length, weight])

toy_pca = PCA(n_components=2).fit(toy)
centre = toy.mean(axis=0)

plt.figure(figsize=(6, 5))
plt.scatter(toy[:, 0], toy[:, 1], s=8, alpha=0.5)
for i, (direction, var) in enumerate(zip(toy_pca.components_, toy_pca.explained_variance_)):
    arrow = direction * 2 * np.sqrt(var)
    plt.annotate("", xy=centre + arrow, xytext=centre, arrowprops=dict(arrowstyle="->", lw=2.5, color="red"))
    plt.text(*(centre + arrow * 1.1), f"PC{i + 1}\n{toy_pca.explained_variance_ratio_[i] * 100:.0f}%", color="red")
plt.xlabel("Body length")
plt.ylabel("Body weight")
plt.axis("equal")
plt.title("PCA finds the directions of largest variation")
plt.show()

### 5.2 Why does PCA reveal population structure?

Imagine two goat populations that have been **isolated** from each other for many generations. Through **genetic drift** (random changes in allele frequency each generation), and perhaps local selection, their allele frequencies drift apart at thousands of SNPs independently. A goat from population A then tends to carry the "A-version" at many SNPs at once, so those SNPs **co-vary** across goats. PCA picks up exactly this kind of shared, correlated signal, so the top PCs usually reflect **shared ancestry**:

* Goats from the same population form a **cluster**.
* Populations that split recently, or still exchange migrants (**gene flow**), are **close together**. Long-isolated populations are **far apart**.
* Individuals with **mixed ancestry** (admixture, crossbreeds) fall roughly **between** their source populations.
* Populations that spread out in space with mostly local migration (**isolation by distance**) often form gradients that look like a map.

### 5.3 PCA in scikit-learn

Most scikit-learn tools follow the same pattern:
```python
model = SomeModel(settings...)          # 1. create the model and choose its settings
result = model.fit_transform(data)      # 2. learn from the data and transform it in one go
model.some_attribute_                   # 3. things learned from the data end with an underscore
```
For PCA, `data` must have the **samples as rows** and the **variables as columns**, which is exactly how our `X_imputed` is organised (goats × SNPs). `PCA` automatically centres each SNP (subtracts its mean) before finding the axes.

### 💻 Code: Run the PCA

Run a PCA on `X_imputed` that keeps the first **10** principal components.

1. Create the model: `pca = PCA(...)`.
2. Fit it and get the PC scores for every goat: `pcs = pca.fit_transform(...)`.
3. Get the % variance explained by each PC: `explained` (as percentages, not fractions).

*Before running: what shape do you expect `pcs` to have?*

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* Useful settings for `PCA(...)`:
  * `n_components=10`: how many PCs to keep.
  * `svd_solver="randomized"`: a fast approximate algorithm for big matrices. The exact one is much slower here.
  * `random_state=42`: the randomized algorithm uses random numbers. Fixing the seed gives the same result every time (any number works).
* `pca.explained_variance_ratio_` holds the fraction of variance per PC. Multiply by 100 for percentages.
* `pcs` has one **row per goat** and one **column per PC**: column 0 is PC1, column 1 is PC2, and so on.

</details>

In [ ]:
check_todo(X_imputed=X_imputed)     # (given) makes sure the imputation TODO is finished

pca = PCA(n_components=10, svd_solver="randomized", random_state=42)
pcs = pca.fit_transform(X_imputed)
explained = pca.explained_variance_ratio_ * 100

check_todo(pcs=pcs, explained=explained)
print("Shape of pcs:", pcs.shape)
print("Variance explained (%):", np.round(explained, 2))

In [ ]:
check_todo(pcs=pcs, explained=explained)
check(pcs.shape == (X.shape[0], 10), "pcs should have one row per goat and 10 columns (one per PC).")
check(abs(explained[0] - 5.5) < 0.3, "PC1 should explain about 5.5%. Did you multiply by 100? Did you use X_imputed?")

### 💻 Code: How much does each PC explain? (scree plot)

Make a **bar plot** with the PC number (1 to 10) on the x-axis and the % variance explained on the y-axis. This is called a **scree plot**. Remember to label your axes!

Look at it: is there a point after which the PCs stop explaining much more variation (an "elbow")?

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* `plt.bar(x_values, heights)` draws a bar plot.
* `range(1, 11)` gives the numbers 1, 2, ..., 10 (the end value is not included!).
* `plt.xlabel("...")` and `plt.ylabel("...")` set the axis labels.

</details>

In [ ]:
plt.figure(figsize=(7, 3))
plt.bar(range(1, 11), explained)
plt.xticks(range(1, 11))
plt.xlabel("Principal component")
plt.ylabel("% variance explained")
plt.show()

## 6. Visualising the population structure

We put the PC scores next to the goat information in one table, so we can colour and filter the plot by breed, country or continent. (This only works because we kept `fam` in sync with `X`!)

In [ ]:
check_todo(pcs=pcs, continent=fam.get("continent", ...))   # needs the PCA TODO and the "Where was each goat sampled?" TODO

pca_df = fam.copy()
for i in range(pcs.shape[1]):
    pca_df[f"PC{i + 1}"] = pcs[:, i]

pca_df.head()

First, a plain plot with all goats in the same colour, like in the paper:

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(pca_df["PC1"], pca_df["PC2"], s=5, alpha=0.5)
plt.xlabel(f"PC1 ({explained[0]:.1f}% variance explained)")
plt.ylabel(f"PC2 ({explained[1]:.1f}% variance explained)")
plt.title("PCA of 4,451 goats")
plt.show()

### ❓ Question: Reading a PCA plot

1. What does each **point** in the plot represent?
2. What do the **axes** represent? Do the units (e.g. "PC1 = 40") mean anything?
3. What does "PC1 explains 5.5% of the variance" mean? Why do you think the percentages are so small?

<details>
<summary><b>✅ Reveal answer</b></summary>

1. Each point is **one goat**.
2. The axes are the first two **principal components**: the two directions through the 48,000-SNP space along which the goats differ the most. A goat's coordinates are its PC scores, a weighted sum of its (centred) genotypes. The absolute values have no biological unit. What matters is **relative position**: goats close together have similar genomes, goats far apart have different genomes. The sign is arbitrary too: running PCA in other software could flip an axis.
3. 5.5% of all the variation among goats, summed over all SNPs, lies along PC1. The percentages are small because most genetic variation is **not** shared population structure. Most of it is variation **within** populations: each goat carries its own unique combination of alleles. On top of that, SNPs on different chromosomes (or far apart) are inherited mostly independently because of recombination, so no single axis can summarise them all. 5–10% for the top PCs is typical (and actually quite strong) for genetic data.

</details>

### 💻 Code: Colour the goats by continent

Remake the plot, but give each **continent** its own colour and add a legend. Put the % variance explained in the axis labels, like in the plot above.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* `df.groupby("column")` splits a table into groups. Looping over it gives you the **name** of each group and the **sub-table**:
  ```python
  for name, group in df.groupby("column"):
      plt.scatter(group["x"], group["y"], label=name)
  ```
  matplotlib automatically picks a new colour for every `plt.scatter` call.
* `label=...` is the text shown in the legend.
* f-strings put values into text: `f"PC1 ({explained[0]:.1f}%)"`. The `:.1f` means "one decimal".

</details>

In [ ]:
plt.figure(figsize=(9, 6))

# loop over the continents and plot each group with its own colour
for continent, group in pca_df.groupby("continent"):
    plt.scatter(group["PC1"], group["PC2"], s=5, alpha=0.6, label=continent)

plt.xlabel(f"PC1 ({explained[0]:.1f}% variance explained)")
plt.ylabel(f"PC2 ({explained[1]:.1f}% variance explained)")
plt.legend(markerscale=4)
plt.show()

### ❓ Question: What do you see, and why?

1. Describe the overall pattern. Which groups does **PC1** separate? And **PC2**?
2. What **evolutionary processes** can create population structure like this?
3. Goats were domesticated about 10,000 years ago in the Near East (the Fertile Crescent: present-day Turkey, Iran, Iraq) and then spread with people into Europe, Africa and Asia. Does the plot fit this history?
4. Compare with Figure 2 in [the paper](https://doi.org/10.1186/s12711-018-0422-x). Do you see the same overall pattern? Why might the exact percentages differ?

<details>
<summary><b>✅ Reveal answer</b></summary>

1. **PC1** mostly separates **European** goats (right) from **African and Asian** goats (left). Goats from the Americas lie in between, and many American breeds are close to the European ones. **PC2** separates the **African** goats (top) from the **Pakistani/Asian** goats (bottom). There are also two compact clusters on the left (around PC2 ≈ −28) containing goats sampled on many continents. We'll find out what they are in the "One breed, many countries" exercise below!
2. Population structure arises when populations **don't mate randomly** with each other:
   * **Genetic drift** in isolated populations makes allele frequencies diverge, faster in small populations.
   * **Founder effects and bottlenecks**: new populations started by a few individuals (e.g. goats brought to a new region) differ by chance from their source.
   * **Geographic isolation and isolation by distance**: animals mate with nearby animals, so distant populations become more different.
   * **Migration / gene flow** works the other way: it makes populations more similar and pulls them together in the plot.
   * **Admixture**: mixing of previously separate populations places individuals between clusters.
   * **Natural selection**, local adaptation (climate, disease), and for goats **artificial selection / breed formation**, which also includes strong drift because breeds are often small and closed.
3. Yes, broadly. The main axes split the goats into the major routes of expansion out of the domestication centre: into Europe, into Africa, and into South/Central Asia. Each branch then drifted apart from the others. North African goats (Morocco, Tunisia) sit between the European and the sub-Saharan African goats, which fits gene flow around the Mediterranean. Be careful though: a PCA alone can't prove a migration route. The picture also mixes ancient history with very recent, human-made movements of modern breeds (see the "One breed, many countries" exercise below).
4. The overall picture should be similar: Europe vs. Africa/Asia on the first axes, and the Americas between Europe and Africa. Exact numbers differ because our analysis is not identical: different QC thresholds (the paper's QC was done in PLINK), a slightly different set of goats and SNPs, whether SNPs in strong linkage disequilibrium were removed ("LD pruning"), whether each SNP was standardised before the PCA, and the software used.

</details>

### 💻 Code: Crossbreeds: test a prediction

Some goats are crossbreeds, e.g. `SAAxCRE` = **Saanen** (`SAA`, a European dairy breed) × **Creole** (`CRE`, an Argentinian local breed).

**First make a prediction:** where in the plot do you expect the `SAAxCRE` goats to be, relative to `SAA` and `CRE`? Why?

Then test it: plot all goats in light grey as background, and on top plot the goats of each breed in `breeds_to_show` in its own colour.

Afterwards, try other combinations, for example `["SEA", "GAL", "SEAxGAL"]` (Small East African × Galla, Kenya) or `["ALP", "SEA", "SEAxALP"]`.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* Loop over the breed names with `for breed in breeds_to_show:`.
* Inside the loop, select the goats of that breed: `subset = pca_df[pca_df["breed"] == breed]`.
* Then `plt.scatter(subset["PC1"], subset["PC2"], label=breed)`.

</details>

In [ ]:
breeds_to_show = ["SAA", "CRE", "SAAxCRE"]

plt.figure(figsize=(9, 6))
plt.scatter(pca_df["PC1"], pca_df["PC2"], s=5, color="lightgrey")   # all goats as background

# plot each breed in breeds_to_show on top, with its own colour and label
for breed in breeds_to_show:
    subset = pca_df[pca_df["breed"] == breed]
    plt.scatter(subset["PC1"], subset["PC2"], s=12, label=breed)

plt.xlabel(f"PC1 ({explained[0]:.1f}%)")
plt.ylabel(f"PC2 ({explained[1]:.1f}%)")
plt.legend(markerscale=2)
plt.show()

### ❓ Question: Crossbreeds

Where are the crossbred goats? Does that match your prediction? Why does this happen?

<details>
<summary><b>✅ Reveal answer</b></summary>

The `SAAxCRE` goats lie **between** the Saanen and Creole clusters. A first-generation cross gets half its genome from each parent breed, so at every SNP its expected genotype is the average of the two parent populations. PCA is a **linear** method: the PC score is a weighted sum of genotypes, so averaging the genotypes also averages the PC scores, putting the cross roughly at the midpoint. Crosses with more ancestry from one parent (e.g. backcrosses) lie closer to that parent. That's why crossbreeds and admixed individuals often form "bridges" between clusters in a PCA.

</details>

### 💻 Code: One breed, many countries

The **Boer** goat (`BOE`) is a meat breed originally developed in South Africa. It is now kept worldwide, and in this dataset it was sampled in 7 countries on 4 continents.

Plot the Boer goats coloured by **country** (on a grey background of all goats again). Do Boer goats from different countries end up in different places?

Then change `"BOE"` to `"ANG"` and do the same for the **Angora** goat, a breed kept for its mohair. Where do the Angora goats end up? Which goats are their neighbours in the plot? (Look up which breeds and countries are around them in `pca_df`, or look up where the Angora breed comes from.)

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

Combine what you did in the last two TODOs: first select the Boer goats (`pca_df[pca_df["breed"] == "BOE"]`), then loop over `.groupby("country")`.

</details>

In [ ]:
boer = pca_df[pca_df["breed"] == "BOE"]

plt.figure(figsize=(9, 6))
plt.scatter(pca_df["PC1"], pca_df["PC2"], s=5, color="lightgrey")
for country, group in boer.groupby("country"):
    plt.scatter(group["PC1"], group["PC2"], s=12, label=country)
plt.xlabel(f"PC1 ({explained[0]:.1f}%)")
plt.ylabel(f"PC2 ({explained[1]:.1f}%)")
plt.legend(markerscale=2)
plt.show()

### ❓ Question: Breed vs. sampling location

What do the Boer and Angora plots tell you? What does this mean when you interpret clusters in the continent-coloured plot?

<details>
<summary><b>✅ Reveal answer</b></summary>

* Boer goats from Switzerland, Australia, the USA, New Zealand, Zimbabwe, Uganda and Tanzania all fall in **the same tight cluster** (bottom left, around PC1 ≈ −30, PC2 ≈ −28). Where a goat lives today says little about its ancestry. What matters is its **breed history**. Commercial breeds have been moved around the world by people very recently, in genetic terms. They haven't had time to drift apart or mix much with local goats.
* The Angora goats, sampled in Argentina, France, South Africa and Madagascar, form the cluster next to it (around PC1 ≈ −17, PC2 ≈ −28). Their neighbours are the goats **sampled in Turkey** (`ANK`, `KIL`, `KLS`) and **Iran** (`IRA`), plus Australian Rangeland and Cashmere goats. The Angora breed comes from the region around Ankara in Turkey (hence the name), so even after being spread worldwide it still sits next to the goats from its region of origin.
* This explains the multi-continent clusters on the left of the continent plot. They are not "Oceanian" or "European" goats in an ancestry sense. They are Boer and Angora goats that happen to have been sampled there.

So colouring by **sampling** location can be misleading. Always ask what the labels really represent. (The paper therefore groups goats by breed and by the geographic origin of the breed.)

</details>

## 7. Optional extra exercises

Finished early, or curious? Try some of these. They are independent of each other.

### 💻 Code: (Optional) PC1 vs. PC3

Plot **PC1 against PC3**, coloured by continent. Then find out which breeds are at the extremes of PC3: compute the **mean PC3 score per breed** and sort it.

What does PC3 separate? Compare with the paper.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* Copy your continent plot and replace `"PC2"` by `"PC3"` (and `explained[1]` by `explained[2]`).
* `pca_df.groupby("breed")["PC3"].mean()` gives the mean PC3 score of every breed. `.sort_values()` sorts it, and `.head()`/`.tail()` show the lowest/highest.

</details>

In [ ]:
plt.figure(figsize=(9, 6))
for continent, group in pca_df.groupby("continent"):
    plt.scatter(group["PC1"], group["PC3"], s=5, alpha=0.6, label=continent)
plt.xlabel(f"PC1 ({explained[0]:.1f}%)")
plt.ylabel(f"PC3 ({explained[2]:.1f}%)")
plt.legend(markerscale=4)
plt.show()

pc3_per_breed = pca_df.groupby("breed")["PC3"].mean().sort_values()
print("Lowest PC3:\n", pc3_per_breed.head(5), "\n\nHighest PC3:\n", pc3_per_breed.tail(5))

### ❓ Question: (Optional) What does PC3 show?

What does PC3 separate?

<details>
<summary><b>✅ Reveal answer</b></summary>

PC3 is essentially a **Boer axis**. The goats with high PC3 scores are almost all **Boer** (`BOE`), followed by Boer crosses (`BOEx`) and some breeds from southern Africa (e.g. from Mozambique), the region where the Boer breed was developed. At the other end are the Pakistani breeds.

Why does one breed get its own axis? There are 300+ Boer goats in the dataset, and as an intensively selected commercial breed they are genetically very similar to each other (strong drift and selection during breed formation). A large, distinct group explains a lot of variance, so PCA gives it an axis. This shows an important property of PCA: **the axes depend on which samples you include and how many**. With fewer Boer goats, PC3 would show something else. Lower PCs often reveal structure *within* the big groups, so it's worth looking beyond PC1 and PC2. Compare with the paper's figures (which used a somewhat different set of goats), and check which breeds are extreme on PC4 or PC5 too!

</details>

### 💻 Code: (Optional) Cumulative variance explained

How much of the total variance do the first 10 PCs explain **together**? Compute the cumulative sum of `explained`.

Based on this, roughly how many PCs do you think you would need to explain 50% of the variance? What does that tell you about genetic variation among goats?

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

`np.cumsum([1, 2, 3])` gives the running total `[1, 3, 6]`.

</details>

In [ ]:
cumulative = np.cumsum(explained)
print(np.round(cumulative, 1))

### ❓ Question: (Optional) How many PCs?

How many PCs would you need to explain 50% of the variance, and what does that tell you?

<details>
<summary><b>✅ Reveal answer</b></summary>

The first 10 PCs together explain only ~17%, and each further PC adds less than 0.5%. You would need **hundreds** of PCs to reach 50%. In other words, most genetic variation among goats is **not** captured by a few major population groups. It is spread over many independent directions: the variation **within** breeds and **between individuals**. This matches a classic result in population genetics (also found in humans): differences *between* populations are real and clearly visible, but they are a small part of all genetic variation. PCA plots look very clustered because they show exactly those few directions where the groups differ most.

</details>

### 💻 Code: (Optional, challenge) How sensitive is the result to the QC thresholds?

Write a function `qc_and_pca(max_missing, min_maf)` that repeats the whole pipeline (load autosomal genotypes → missingness filters → MAF filter → mean imputation → PCA with 2 PCs) and returns the PC scores, the explained variance (%) and the remaining goat information.

Use it to compare the default (`0.05, 0.05`) with stricter settings, e.g. `max_missing=0.01` and/or `min_maf=0.2`. How do the number of goats and SNPs, the % variance explained and the plot change?

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* Start from a fresh copy of the data inside the function (re-read the `.bed` and `.fam` files), so you don't overwrite `X` and `fam`.
* Then copy-paste your solutions from Sections 3–5, replacing `0.05` by the function arguments.
* End with `return scores, explained, info`.
* `fig, axes = plt.subplots(1, 3, figsize=(21, 6))` makes three plots side by side. Use `axes[0].scatter(...)`, `axes[1].scatter(...)`, ... to draw in each.
* To give each continent the same colour in every panel, pass `color="C0"`, `"C1"`, ... yourself (matplotlib's default colours) instead of letting matplotlib choose.

</details>

In [ ]:
def qc_and_pca(max_missing, min_maf):
    G = open_bed(data_dir / "snps.bed").read(index=np.s_[:, is_autosome], dtype="float32")
    info = pd.read_csv(data_dir / "snps.fam", sep=r"\s+", header=None,
                       names=["breed", "iid", "father", "mother", "sex", "phenotype"])
    info["continent"] = info["iid"].str[:2].map(country_to_continent)

    keep_goats = np.isnan(G).mean(axis=1) <= max_missing
    G, info = G[keep_goats], info[keep_goats].reset_index(drop=True)
    G = G[:, np.isnan(G).mean(axis=0) <= max_missing]
    p = np.nanmean(G, axis=0) / 2
    G = G[:, np.minimum(p, 1 - p) >= min_maf]
    print(f"max_missing={max_missing}, min_maf={min_maf}: {G.shape[0]} goats x {G.shape[1]} SNPs")

    G = np.where(np.isnan(G), np.nanmean(G, axis=0), G)
    model = PCA(n_components=2, svd_solver="randomized", random_state=42)
    scores = model.fit_transform(G)
    return scores, model.explained_variance_ratio_ * 100, info


all_continents = sorted(set(country_to_continent.values()))   # fixed order -> same colour per continent in every panel
settings = [(0.05, 0.05), (0.05, 0.2), (0.01, 0.05)]

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (max_missing, min_maf) in zip(axes, settings):
    scores, var, info = qc_and_pca(max_missing, min_maf)
    for i, continent in enumerate(all_continents):
        sel = (info["continent"] == continent).to_numpy()
        ax.scatter(scores[sel, 0], scores[sel, 1], s=5, alpha=0.6, color=f"C{i}", label=continent)
    ax.set_title(f"max_missing={max_missing}, min_maf={min_maf}\n{len(info)} goats")
    ax.set_xlabel(f"PC1 ({var[0]:.1f}%)")
    ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")
axes[0].legend(markerscale=4)
plt.show()

### ❓ Question: (Optional) Effect of stricter QC

What changed when you made the filters stricter?

<details>
<summary><b>✅ Reveal answer</b></summary>

* **Stricter MAF filtering** (`min_maf=0.2`) keeps fewer, only common SNPs, but the plot looks practically the same. Common SNPs carry most of the between-population signal, so the big picture is **robust** to this choice. (You do lose SNPs that are common in only a few breeds, which can matter for separating closely related breeds.)
* **Stricter missingness filtering** (`max_missing=0.01`) has a dramatic effect: only ~900 of the ~4,450 goats survive, and whole groups disappear: all goats from the Americas and Oceania, and every European country except Italy. The plot changes completely, and PC1 now explains more variance because there are fewer, more distinct groups left.
* Why? The surviving goats come from only 8 countries (Pakistan, Italy, Mali, Zimbabwe, Madagascar, Cameroon, South Africa and Tanzania), where the samples have very low missingness. That is very likely a **technical batch effect** (different labs, DNA extraction methods or genotyping runs), not biology. A threshold that is too strict therefore doesn't just clean the data: it changes **which populations you study**.
* Take-home: always check how sensitive your conclusions are to analysis choices, and look at **who** gets removed by a filter, not only how many.

</details>

### 💻 Code: (Optional) A PCA of the X chromosome only

Repeat the analysis using **only the X-chromosome SNPs** (chromosome `30`) and plot PC1 vs. PC2 coloured by continent. Use the original (unfiltered) `.bim` file to build the mask, because our `bim` now only contains autosomal SNPs.

How does it compare to the autosomal PCA? Which problems from the "Why only autosomes?" question might show up here?

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* Read the `.bim` file again into a new variable, and make the mask `(bim_all["chrom"] == 30).to_numpy()`.
* To read only the goats that passed QC, you need their **row numbers** in the `.bed` file:
  `goat_rows = np.flatnonzero(np.isin(bed.iid, fam["iid"]))`. `np.isin` gives True for the IDs that are in `fam`, and `np.flatnonzero` turns True/False into positions.
* Then `bed.read(index=np.s_[goat_rows, is_x], dtype="float32")`.
* Apply the SNP filters and the imputation as before, and run a PCA.

</details>

In [ ]:
bim_all = pd.read_csv(data_dir / "snps.bim", sep=r"\s+", header=None,
                      names=["chrom", "snp", "cM", "pos", "allele_1", "allele_2"])
is_x = (bim_all["chrom"] == 30).to_numpy()

# same goats as in our main analysis: select the rows of the goats that passed QC
goat_rows = np.flatnonzero(np.isin(bed.iid, fam["iid"]))
X_chrX = bed.read(index=np.s_[goat_rows, is_x], dtype="float32")

p_x = np.nanmean(X_chrX, axis=0) / 2
X_chrX = X_chrX[:, (np.isnan(X_chrX).mean(axis=0) <= 0.05) & (np.minimum(p_x, 1 - p_x) >= 0.05)]
X_chrX = np.where(np.isnan(X_chrX), np.nanmean(X_chrX, axis=0), X_chrX)
print("X-chromosome matrix:", X_chrX.shape)

pca_x = PCA(n_components=2, svd_solver="randomized", random_state=42)
pcs_x = pca_x.fit_transform(X_chrX)
var_x = pca_x.explained_variance_ratio_ * 100

plt.figure(figsize=(9, 6))
for continent in sorted(fam["continent"].unique()):
    sel = (fam["continent"] == continent).to_numpy()
    plt.scatter(pcs_x[sel, 0], pcs_x[sel, 1], s=5, alpha=0.6, label=continent)
plt.xlabel(f"PC1 ({var_x[0]:.1f}%)")
plt.ylabel(f"PC2 ({var_x[1]:.1f}%)")
plt.title("PCA of X-chromosome SNPs only")
plt.legend(markerscale=4)
plt.show()

### ❓ Question: (Optional) X chromosome

How does the X-chromosome PCA compare to the autosomal one?

<details>
<summary><b>✅ Reveal answer</b></summary>

The broad continental pattern is usually still visible, because the X shares the same population history. But the picture is **noisier**, for several reasons:
* there are far fewer SNPs (~2,000 instead of ~48,000), so there's more random noise;
* males have only one X, so they appear homozygous at every X SNP (here often coded as 0 or 2, or as missing). Their genotypes don't mean the same as females'. Since sex is unknown for most goats, we can't analyse the sexes separately;
* the X has a smaller effective population size (¾ of the autosomes), so it drifts faster. It is also shaped by sex-biased processes: if mostly males migrated between populations, the X would show **less** gene flow than the autosomes;
* part of the X (the pseudo-autosomal region) behaves like an autosome, which makes things more complicated still.

So differences between X and autosomal PCAs can be biologically interesting (e.g. sex-biased migration), but they are hard to interpret without knowing the sex of each animal.

</details>

### 💻 Code: (Optional) Does the imputation strategy matter?

Instead of the mean, impute each missing genotype with the **most common genotype** at that SNP. scikit-learn's `SimpleImputer` can do this for you. Run the PCA again and compare the explained variance and the plot with the mean-imputed version.

<details>
<summary><b>💡 Syntax explained (click to open)</b></summary>

* `SimpleImputer(strategy="most_frequent").fit_transform(X)` returns a copy of `X` with every `NaN` replaced by the most common value of its **column** (SNP). Other strategies: `"mean"`, `"median"`.
* Then run `PCA(...).fit_transform(...)` on the result, just like before.
* (Note: the signs of the PCs are arbitrary, so the whole plot may come out mirrored. That doesn't mean anything.)

</details>

In [ ]:
from sklearn.impute import SimpleImputer

X_mode = SimpleImputer(strategy="most_frequent").fit_transform(X)
pcs_mode = PCA(n_components=2, svd_solver="randomized", random_state=42).fit_transform(X_mode)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, scores, title in [(axes[0], pcs, "mean imputation"), (axes[1], pcs_mode, "most-frequent imputation")]:
    for continent in sorted(pca_df["continent"].unique()):
        sel = (pca_df["continent"] == continent).to_numpy()
        ax.scatter(scores[sel, 0], scores[sel, 1], s=5, alpha=0.6, label=continent)
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
axes[0].legend(markerscale=4)
plt.show()

### ❓ Question: (Optional) Imputation strategy

Did the imputation strategy change the result?

<details>
<summary><b>✅ Reveal answer</b></summary>

The two plots are practically identical (apart from a possible mirror flip, since PC signs are arbitrary). Only ~0.3% of the genotypes were imputed, so how we fill them in hardly matters. Imputation strategy matters much more with lots of missing data. Mode imputation then pushes values towards the most common homozygote. Mean imputation pulls goats towards the centre of the plot. Population-aware or LD-based imputation gives the best guesses. That's the real reason we did the missingness QC first: it keeps the amount of guessing small.

</details>

## 8. A word of caution

PCA is quick, easy and makes beautiful plots, which is exactly why it is easy to over-interpret. The positions in a PCA plot depend on **which samples you include** (e.g. many goats from one breed can "pull" an axis towards that breed), on QC choices, and on scaling. A PCA shows **similarity**; it does not by itself tell you *why* populations are similar (shared ancestry? gene flow? the same founders?) or in which direction migration happened. Combine it with other methods (e.g. ADMIXTURE, f-statistics, phylogenetic trees) and with what you know about the history of the populations.

For a critical discussion, see [Elhaik (2022), *Scientific Reports*: "Principal Component Analyses (PCA)-based findings in population genetic studies are highly biased and must be reevaluated"](https://www.nature.com/articles/s41598-022-14395-4).